# Week 7, Day 2: Asking the base model for a price

**Lesson:** [the Colab notebook](https://colab.research.google.com/drive/1wO3lNMrMfprlJZF4X9fSsQ8tYC3SRZbh) linked from [week7/day2.ipynb](../../../week7/day2.ipynb), *Prompt data and base model*. **About 25 minutes.**

**What you practice here**

- Choosing a token cutoff from the summaries' token counts, so that only a few summaries get cut.
- Loading the tokenizer and the 4-bit base model for inference with `AutoTokenizer` and `AutoModelForCausalLM`, with the pad token set and the model's generation config told about it.
- Writing the lesson's `model_predict` yourself: the tokenizer's tensors moved to the GPU, `generate` with a limit of 8 new tokens, and only the new tokens decoded.
- Scoring the base model with the course's `evaluate`, in Try it for real.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week7/day2", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson measures the product summaries in tokens and picks a cutoff of 110 tokens. Then, in Colab, it loads
Llama 3.2 3B in 4 bits and asks it for prices, to see how the base model does before any fine-tuning. Do the same,
and ask the base model about wrestling merchandise too: a replica title belt, an event poster, a signed photo.

transformers is not in the checks' Python, so `load_for_inference` takes the two auto classes as parameters, and
`predict` takes the model and the tokenizer; they call them as the lesson does. The checks hand them stand-ins: a
loader, a tokenizer that gives token ids as tensors, and a model whose `generate` answers with a canned price.

## What each call should give

```python
pick_cutoff([12, 35, 48, 51, 95, 101, 130], max_share=0.15)
# 110: only the summary of 130 tokens is longer, 1 in 7

tokenizer, model = load_for_inference(AutoTokenizer, AutoModelForCausalLM, quant_config)
# the Llama 3.2 3B tokenizer, padding with its eos token on the right, and the model in 4 bits

BELT = ("What does this cost to the nearest dollar?\n\nTitle: WWE Championship Replica Title Belt\n"
        "Category: Collectibles\nBrand: WWE\nDescription: A full-size replica of the WWE Championship title belt.\n"
        "Details: Zinc-alloy plates on a genuine leather strap, with a carry bag.\n\nPrice is $")
predict(model, tokenizer, BELT)
# the new tokens only, such as '499.00<|end_of_text|>': the course's evaluate reads the number from it
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `pick_cutoff(token_counts, max_share=0.05, step=10)`

Picks the smallest round cutoff that cuts at most a given share of the summaries. `token_counts` is a list of ints,
one per summary; in the real run they come from the course's `item.count_tokens(tokenizer)`. A summary is cut when
it has more tokens than the cutoff, as in the lesson's count. Return the smallest of `step`, `2 * step`,
`3 * step`, and so on, for which the share of counts over it is at most `max_share`. Raise `ValueError` if
`token_counts` is empty.

<details><summary>Hint</summary>

Start at `step`, and add `step` while too many are cut. The lesson counts them as
`len([count for count in token_counts if count > CUTOFF])`, and the share is that count over `len(token_counts)`.

</details>

In [ ]:
def pick_cutoff(token_counts, max_share=0.05, step=10):
    # Write pick_cutoff here, as described above.
    raise NotImplementedError("pick_cutoff is not written yet")

In [ ]:
check("pick_cutoff")

### 2 · `load_for_inference(auto_tokenizer, auto_model, quant_config, name="meta-llama/Llama-3.2-3B")`

Loads the tokenizer and the quantized model as the lesson does for inference, and returns both. `auto_tokenizer` and
`auto_model` are `AutoTokenizer` and `AutoModelForCausalLM`, and `quant_config` is a ready `BitsAndBytesConfig`.

- Load the tokenizer of `name`. Make it pad with its end-of-sequence token, and pad on the right.
- Load the model `name` with `quant_config` as its quantization config, and let the loader place it on the GPU, as
  the lesson's `device_map="auto"` does. Then give the model's generation config the tokenizer's pad token id,
  once the tokenizer has its pad token.
- Return `(tokenizer, model)`.

<details><summary>Hint</summary>

```python
tokenizer = auto_tokenizer.from_pretrained(name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model = auto_model.from_pretrained(name, quantization_config=quant_config, device_map="auto")
model.generation_config.pad_token_id = tokenizer.pad_token_id
```

</details>

In [ ]:
def load_for_inference(auto_tokenizer, auto_model, quant_config, name="meta-llama/Llama-3.2-3B"):
    # Write load_for_inference here, as described above.
    raise NotImplementedError("load_for_inference is not written yet")

In [ ]:
check("load_for_inference")

### 3 · `predict(model, tokenizer, prompt, device="cuda", max_new_tokens=8)`

Asks the model to continue a prompt, and returns only the text it adds. This is the lesson's `model_predict`, on a
prompt string. Tokenize the prompt as PyTorch tensors, move them to `device`, and generate at most `max_new_tokens`
new tokens, passing the attention mask along with the ids, as unpacking the tokenizer's output does.

`generate` gives back the prompt's ids followed by the new ones. Return the decoded text of the new ones only, from
the first row. Dropping the end-of-text token as you decode is fine. You may leave out the lesson's
`with torch.no_grad():`, as `generate` already runs without gradients.

<details><summary>Hint</summary>

```python
inputs = tokenizer(prompt, return_tensors="pt").to(device)
output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
prompt_len = inputs["input_ids"].shape[1]
return tokenizer.decode(output_ids[0, prompt_len:])
```

</details>

In [ ]:
def predict(model, tokenizer, prompt, device="cuda", max_new_tokens=8):
    # Write predict here, as described above.
    raise NotImplementedError("predict is not written yet")

In [ ]:
check("predict")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 7, Day 2 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. This needs a GPU, so it runs in Google Colab: push this notebook, then [open it in Colab](https://colab.research.google.com/github/starlest/llm_engineering/blob/main/plan/exercises/week7/day2.ipynb), choose a T4 runtime, and run the cells from the top. The first cell fetches the checker from your fork, so the checks run in Colab too. Write the steps in the 2 cells at the end.

Colab works on its own copy of the notebook. To keep the Try code you write there, use **File > Save a copy in GitHub** (to this notebook's path in your fork), then run `git pull` in your codespace before you commit there again. Your parts are already in your fork, which is what the tracker reads.

Colab's free T4 is enough for Llama 3.2 3B in 4 bits. Accept the Llama 3.2 license on its Hugging Face page, and add
your token to Colab's secrets as `HF_TOKEN`. Scoring 200 test items takes a few minutes.

1. Keep the `!pip install` and `!wget` lines in the first Try cell: they fetch bitsandbytes and the course's
   `util.py`. Log in to Hugging Face with your `HF_TOKEN` secret.
2. Build the lesson's 4-bit `BitsAndBytesConfig`, and call `load_for_inference` with `AutoTokenizer` and
   `AutoModelForCausalLM`.
3. Count the tokens of each summary in `ed-donner/items_lite` with
   `tokenizer.encode(summary, add_special_tokens=False)`, and compare `pick_cutoff(counts)` with the lesson's 110.
4. Load the test split of `ed-donner/items_prompts_lite` as `test`, and print `predict` for `test[0]["prompt"]`
   beside `test[0]["completion"]`. Then ask about a wrestling item of your own, written in the same format as those
   prompts: the question, then five lines (Title, Category, Brand, Description and Details), then `Price is $`.
5. Wrap `predict` in a function of one datapoint, `base_llama(item)`, and score it with `evaluate(base_llama, test)`,
   imported from `util`. Compare its error with the 110.72 in `week7/results.ipynb`; the chart takes its title from
   the function's name.

<details><summary>Stuck? One way to write it</summary>

```python
!pip install -q --upgrade bitsandbytes
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py
import torch
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

login(userdata.get("HF_TOKEN"), add_to_git_credential=True)
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                  bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
tokenizer, model = load_for_inference(AutoTokenizer, AutoModelForCausalLM, quant_config)

summaries = load_dataset("ed-donner/items_lite", split="train")["summary"]
counts = [len(tokenizer.encode(summary, add_special_tokens=False)) for summary in summaries]
print("The smallest cutoff that cuts at most 5% of the summaries:", pick_cutoff(counts))
```

```python
from util import evaluate

test = load_dataset("ed-donner/items_prompts_lite", split="test")
print(predict(model, tokenizer, test[0]["prompt"]), "| truth:", test[0]["completion"])
belt = ("What does this cost to the nearest dollar?\n\nTitle: WWE Championship Replica Title Belt\n"
        "Category: Collectibles\nBrand: WWE\nDescription: A full-size replica of the WWE Championship title belt.\n"
        "Details: Zinc-alloy plates on a genuine leather strap, with a carry bag.\n\nPrice is $")
print(predict(model, tokenizer, belt))


def base_llama(item):
    return predict(model, tokenizer, item["prompt"])


evaluate(base_llama, test)
```

</details>

In [ ]:
!pip install -q --upgrade bitsandbytes
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py
# Steps 1 to 3: log in, load the tokenizer and the 4-bit model, and pick a cutoff

In [ ]:
# Steps 4 and 5: ask for prices, then score the base model with evaluate